# LSTM 的數學推導

> 目標：先簡單複習 LSTM 的 forward，再從 Output Layer 開始做 BPTT（Backpropagation Through Time）完整推導。  
> 記號以向量形式表示，$\odot$ 表示逐元素相乘。


<img src="image/lstm.webp" alt="lstm" style="width: 70%;"/>

<img src="image/lstm-time-series.webp" alt="lstm in time series" style="width: 70%;"/>

圖片來源： https://www.geeksforgeeks.org/deep-learning/deep-learning-introduction-to-long-short-term-memory/

---

## 0) 符號定義

在時間步 $t$：

- 輸入：$x_t$
- 前一個 hidden state：$h_{t-1}$
- 前一個 cell state：$C_{t-1}$

若採用「分開權重」寫法（你指定的形式）：

- forget gate: $(W_f, U_f, b_f)$
- input gate: $(W_i, U_i, b_i)$
- output gate: $(W_o, U_o, b_o)$
- candidate cell: $(W_c, U_c, b_c)$

輸出層：

- $(W_y, b_y)$

---

## 1) LSTM Forward（向前傳播）

### 1.1 Forget gate
$$
f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)
$$

### 1.2 Input gate
$$
i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)
$$

### 1.3 Output gate
$$
o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)
$$

### 1.4 Candidate cell（候選記憶）
$$
\tilde{C}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)
$$

### 1.5 Cell state 更新（核心）
$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
$$

### 1.6 Hidden state（輸出）
$$
h_t = o_t \odot \tanh(C_t)
$$

### 1.7 Output layer（以分類為例）
先定義 logits：
$$
z_t = W_y h_t + b_y
$$

機率輸出：
$$
y_t = \mathrm{softmax}(z_t)
$$

若真實標籤為 one-hot 向量 $\hat{y}_t$，則交叉熵 loss：
$$
L_t = -\sum_k \hat{y}_{t,k}\log y_{t,k}
$$

總 loss：
$$
L = \sum_{t=1}^{T} L_t
$$

---

## 2) 會用到的基本導數

### 2.1 Sigmoid 導數
$$
\sigma'(x) = \sigma(x)\bigl(1-\sigma(x)\bigr)
$$

### 2.2 Tanh 導數
$$
\frac{d}{dx}\tanh(x) = 1-\tanh^2(x)
$$

---


## 3) 從 Output Layer 開始 Backward（BPTT）

### 3.1 Softmax + Cross Entropy 的梯度

定義：
$$
\hat{y}_t = \mathrm{softmax}(z_t), \quad
L_t = -\sum_i y_{t,i} \log \hat{y}_{t,i}
$$

令：
$$
\delta^y_t \equiv \frac{\partial L_t}{\partial z_t}
$$

則有標準結果：
$$
\delta^y_t = \hat{y}_t - y_t
$$

---

### 3.2 對輸出層參數與 hidden 的梯度

假設輸出層為：
$$
z_t = W_y h_t + b_y
$$

其中：
- $h_t \in \mathbb{R}^{H \times 1}$
- $z_t \in \mathbb{R}^{V \times 1}$
- $W_y \in \mathbb{R}^{V \times H}$

則：

對 $W_y$：
$$
\frac{\partial L_t}{\partial W_y}
= \delta^y_t h_t^T
$$

對 $b_y$：
$$
\frac{\partial L_t}{\partial b_y}
= \delta^y_t
$$

對 $h_t$（來自當前時間步輸出層）：
$$
\frac{\partial L_t}{\partial h_t}
= W_y^T \delta^y_t
$$

---

### 3.3 BPTT 中 hidden state 的總梯度

整段序列的總 loss 定義為：
$$
L = \sum_{k=1}^T L_k
$$

在時間步 $t$，hidden state 的總梯度定義為：
$$
\delta^h_t \equiv \frac{\partial L}{\partial h_t}
$$

由於 $h_t$ 不只影響當前的 $L_t$，也會透過 recurrent connection 影響未來時間步，
因此可以分解為：

$$
\delta^h_t=
\underbrace{\frac{\partial L_t}{\partial h_t}}_{\text{當前輸出}}
+
\underbrace{\sum_{k>t} \frac{\partial L_k}{\partial h_t}}_{\text{未來時間步}}
$$

整理後可寫成：
$$
\delta^h_t = W_y^T \delta^y_t + \delta^{h,\text{future}}_t
$$

其中：
- 第一項來自當前時間步的輸出層
- 第二項為從未來時間步反傳回來的梯度

---

### 3.4 LSTM 的關鍵補充（與普通 RNN 的差異）

在 LSTM 中，除了 hidden state $h_t$ 之外，cell state $c_t$ 也會影響未來時間步。

因此在反向傳播中，除了：
$$
\delta^h_t = \frac{\partial L}{\partial h_t}
$$

還必須定義：
$$
\delta^c_t = \frac{\partial L}{\partial c_t}
$$

並且需要同時考慮兩條梯度路徑：
- $h_t \rightarrow h_{t+1} \rightarrow \cdots$
- $c_t \rightarrow c_{t+1} \rightarrow \cdots$

其中第二條（cell state）是 LSTM 能有效緩解 gradient vanishing 的關鍵。

## 4) 從 $h_t$ 回傳到 $o_t$ 與 $c_t$

由
$$
h_t = o_t \odot \tanh(c_t)
$$

### 4.1 對 output gate 的梯度

$$
\frac{\partial L}{\partial o_t}=
\frac{\partial L}{\partial h_t}
\odot
\frac{\partial h_t}{\partial o_t}=
\delta_t^h \odot \tanh(c_t)
$$

定義
$$
\delta_t^o \equiv \frac{\partial L}{\partial o_t}
$$

因此
$$
\delta_t^o = \delta_t^h \odot \tanh(c_t)
$$

### 4.2 由 $h_t$ 傳到 $c_t$ 的梯度（局部貢獻）

首先，
$$
\frac{\partial h_t}{\partial c_t}=
o_t \odot \bigl(1 - \tanh^2(c_t)\bigr)
$$

因此，經由 $h_t$ 傳遞到 $c_t$ 的梯度貢獻為
$$
\delta_t^{c,(h)}
\equiv
\left.\frac{\partial L}{\partial c_t}\right|_{\text{via } h_t}=
\delta_t^h \odot o_t \odot \bigl(1 - \tanh^2(c_t)\bigr)
$$

### 4.3 $c_t$ 的總梯度

$c_t$ 的梯度除了來自 $h_t$，還包含來自未來時間步的回傳：

$$
\delta_t^c
\equiv
\frac{\partial L}{\partial c_t}=
\delta_t^{c,(h)}
+
\delta_{t+1}^c \odot f_{t+1}
$$

亦即

$$
\delta_t^c=
\delta_t^h \odot o_t \odot \bigl(1 - \tanh^2(c_t)\bigr)
+
\delta_{t+1}^c \odot f_{t+1}
$$

## 5) Cell state 的總梯度（LSTM 最關鍵）

先回憶 LSTM 在時間步 $t$ 的 hidden state：
$$
h_t = o_t \odot \tanh(c_t)
$$

因此，從 $h_t$ 回傳到 $c_t$ 的梯度為：
$$
\delta^h_t \odot o_t \odot \bigl(1-\tanh^2(c_t)\bigr)
$$

另一方面，由於 cell state 會沿時間傳遞：
$$
c_{t+1} = f_{t+1}\odot c_t + i_{t+1}\odot \tilde{c}_{t+1}
$$

所以 $c_t$ 的總梯度除了來自當前 $h_t$，還包含來自未來 $c_{t+1}$ 的梯度。

定義：
$$
\delta^c_t \equiv \frac{\partial L}{\partial c_t}
$$

則：
$$
\delta^c_t=
\frac{\partial L}{\partial h_t} \odot \frac{\partial h_t}{\partial c_t}+
\frac{\partial L}{\partial c_{t+1}} \odot \frac{\partial c_{t+1}}{\partial c_t }=
\delta^h_t \odot o_t \odot \bigl(1-\tanh^2(c_t)\bigr)
+
\delta^c_{t+1}\odot f_{t+1}
$$

這條就是 LSTM BPTT 的核心式之一。

---

## 6) 從 $h_t$ 與 $c_t$ 分流到各 gate

由
$$
h_t = o_t \odot \tanh(c_t)
$$

可先得到 output gate 的梯度：

### 6.1 Output gate 的梯度
$$
\delta^o_t \equiv \frac{\partial L}{\partial o_t} = \delta^h_t \odot \tanh(c_t)
$$

接著，由
$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

可得其餘分支：

### 6.2 Forget gate 的梯度
$$
\delta^f_t \equiv \frac{\partial L}{\partial f_t} = \delta^c_t \odot c_{t-1}
$$

### 6.3 Input gate 的梯度
$$
\delta^i_t \equiv \frac{\partial L}{\partial i_t} = \delta^c_t \odot \tilde{c}_t
$$

### 6.4 Candidate cell 的梯度
$$
\delta^{\tilde{c}}_t \equiv \frac{\partial L}{\partial \tilde{c}_t} = \delta^c_t \odot i_t
$$

### 6.5 對前一個 cell state 的梯度
$$
\delta^c_{t-1} = \delta^c_t \odot f_t
$$

注意：這裡的 $\delta^c_{t-1}$ 是從時間步 $t$ 傳回去的 cell-state 梯度分量；在實作 BPTT 時，通常會把它作為下一輪迴圈中的 future cell gradient 使用。

---

## 7) 回到各 gate 的 pre-activation（乘上 activation 導數）

定義各 gate 的 pre-activation：
$$
a^f_t = W_f x_t + U_f h_{t-1} + b_f
$$
$$
a^i_t = W_i x_t + U_i h_{t-1} + b_i
$$
$$
a^o_t = W_o x_t + U_o h_{t-1} + b_o
$$
$$
a^c_t = W_c x_t + U_c h_{t-1} + b_c
$$

且
$$
f_t=\sigma(a^f_t),\quad i_t=\sigma(a^i_t),\quad o_t=\sigma(a^o_t),\quad \tilde{c}_t=\tanh(a^c_t)
$$

因此：

### 7.1 Forget gate pre-activation 梯度
$$
\delta^{a_f}_t \equiv \frac{\partial L}{\partial a^f_t}
= \delta^f_t \odot f_t \odot (1-f_t)
$$

### 7.2 Input gate pre-activation 梯度
$$
\delta^{a_i}_t \equiv \frac{\partial L}{\partial a^i_t}
= \delta^i_t \odot i_t \odot (1-i_t)
$$

### 7.3 Output gate pre-activation 梯度
$$
\delta^{a_o}_t \equiv \frac{\partial L}{\partial a^o_t}
= \delta^o_t \odot o_t \odot (1-o_t)
$$

### 7.4 Candidate cell pre-activation 梯度
$$
\delta^{a_c}_t \equiv \frac{\partial L}{\partial a^c_t}
= \delta^{\tilde{c}}_t \odot \bigl(1-\tilde{c}_t^2\bigr)
$$

---



## 8) 對參數 $W, U, b$ 的梯度

整段序列的總 loss 為
$$
L = \sum_{t=1}^T L_t
$$

因此各參數梯度需對所有時間步加總。

### 8.1 Forget gate 參數
$$
\frac{\partial L}{\partial W_f} = \sum_t \delta^{a_f}_t x_t^T
$$
$$
\frac{\partial L}{\partial U_f} = \sum_t \delta^{a_f}_t h_{t-1}^T
$$
$$
\frac{\partial L}{\partial b_f} = \sum_t \delta^{a_f}_t
$$

### 8.2 Input gate 參數
$$
\frac{\partial L}{\partial W_i} = \sum_t \delta^{a_i}_t x_t^T
$$
$$
\frac{\partial L}{\partial U_i} = \sum_t \delta^{a_i}_t h_{t-1}^T
$$
$$
\frac{\partial L}{\partial b_i} = \sum_t \delta^{a_i}_t
$$

### 8.3 Output gate 參數
$$
\frac{\partial L}{\partial W_o} = \sum_t \delta^{a_o}_t x_t^T
$$
$$
\frac{\partial L}{\partial U_o} = \sum_t \delta^{a_o}_t h_{t-1}^T
$$
$$
\frac{\partial L}{\partial b_o} = \sum_t \delta^{a_o}_t
$$

### 8.4 Candidate cell 參數
$$
\frac{\partial L}{\partial W_c} = \sum_t \delta^{a_c}_t x_t^T
$$
$$
\frac{\partial L}{\partial U_c} = \sum_t \delta^{a_c}_t h_{t-1}^T
$$
$$
\frac{\partial L}{\partial b_c} = \sum_t \delta^{a_c}_t
$$

### 8.5 輸出層參數
$$
\frac{\partial L}{\partial W_y} = \sum_t \delta^y_t h_t^T
$$
$$
\frac{\partial L}{\partial b_y} = \sum_t \delta^y_t
$$

---

## 9) 往回傳到 $h_{t-1}$ 與 $x_t$

由於四個 gate 的 pre-activation 都依賴 $h_{t-1}$ 和 $x_t$，所以梯度需加總。

### 9.1 從時間步 $t$ 傳回到 $h_{t-1}$ 的梯度
$$
\delta^{h,\text{future}}_{t-1}=
U_f^T\delta^{a_f}_t
+
U_i^T\delta^{a_i}_t
+
U_o^T\delta^{a_o}_t
+
U_c^T\delta^{a_c}_t
$$

因此在時間步 $t-1$，總 hidden gradient 為：
$$
\delta^h_{t-1}=
W_y^T\delta^y_{t-1}
+
\delta^{h,\text{future}}_{t-1}
$$

### 9.2 對輸入 $x_t$ 的梯度
$$
\frac{\partial L}{\partial x_t}=
W_f^T\delta^{a_f}_t
+
W_i^T\delta^{a_i}_t
+
W_o^T\delta^{a_o}_t
+
W_c^T\delta^{a_c}_t
$$

---

## 10) BPTT 的時間迴圈（從 $T \to 1$）

初始化：
$$
\delta^{h,\text{future}}_T = 0,\qquad \delta^c_{T+1}=0
$$

對每個時間步 $t=T,T-1,\dots,1$：

### Step 1. Output layer
$$
\delta^y_t = \hat{y}_t - y_t
$$
$$
\delta^h_t = W_y^T\delta^y_t + \delta^{h,\text{future}}_t
$$

### Step 2. Hidden $\to$ Output gate / Cell
由
$$
h_t = o_t \odot \tanh(c_t)
$$
得
$$
\delta^o_t = \delta^h_t \odot \tanh(c_t)
$$
以及
$$
\delta^c_t=
\delta^h_t \odot o_t \odot \bigl(1-\tanh^2(c_t)\bigr)
+
\delta^c_{t+1}\odot f_{t+1}
$$

### Step 3. Cell 分流
$$
\delta^f_t = \delta^c_t \odot c_{t-1}
$$
$$
\delta^i_t = \delta^c_t \odot \tilde{c}_t
$$
$$
\delta^{\tilde{c}}_t = \delta^c_t \odot i_t
$$
$$
\delta^c_{t-1} = \delta^c_t \odot f_t
$$

### Step 4. 經 activation 導數
$$
\delta^{a_f}_t = \delta^f_t \odot f_t \odot (1-f_t)
$$
$$
\delta^{a_i}_t = \delta^i_t \odot i_t \odot (1-i_t)
$$
$$
\delta^{a_o}_t = \delta^o_t \odot o_t \odot (1-o_t)
$$
$$
\delta^{a_c}_t = \delta^{\tilde{c}}_t \odot (1-\tilde{c}_t^2)
$$

### Step 5. 累加參數梯度
$$
\nabla W_f \mathrel{+}= \delta^{a_f}_t x_t^T,\quad
\nabla U_f \mathrel{+}= \delta^{a_f}_t h_{t-1}^T,\quad
\nabla b_f \mathrel{+}= \delta^{a_f}_t
$$
$$
\nabla W_i \mathrel{+}= \delta^{a_i}_t x_t^T,\quad
\nabla U_i \mathrel{+}= \delta^{a_i}_t h_{t-1}^T,\quad
\nabla b_i \mathrel{+}= \delta^{a_i}_t
$$
$$
\nabla W_o \mathrel{+}= \delta^{a_o}_t x_t^T,\quad
\nabla U_o \mathrel{+}= \delta^{a_o}_t h_{t-1}^T,\quad
\nabla b_o \mathrel{+}= \delta^{a_o}_t
$$
$$
\nabla W_c \mathrel{+}= \delta^{a_c}_t x_t^T,\quad
\nabla U_c \mathrel{+}= \delta^{a_c}_t h_{t-1}^T,\quad
\nabla b_c \mathrel{+}= \delta^{a_c}_t
$$
$$
\nabla W_y \mathrel{+}= \delta^y_t h_t^T,\quad
\nabla b_y \mathrel{+}= \delta^y_t
$$

### Step 6. 傳給前一 hidden
$$
\delta^{h,\text{future}}_{t-1}=
U_f^T\delta^{a_f}_t
+
U_i^T\delta^{a_i}_t
+
U_o^T\delta^{a_o}_t
+
U_c^T\delta^{a_c}_t
$$

### Step 7. 傳給前一 cell
$$
\delta^c_{t-1} = \delta^c_t \odot f_t
$$

---

## 11) 一句話總結（核心）

LSTM 的 BPTT 可以濃縮成兩條最重要的傳遞式：

### 11.1 Hidden state 的總梯度
$$
\delta^h_t = W_y^T\delta^y_t + \delta^{h,\text{future}}_t
$$

其中
$$
\delta^{h,\text{future}}_t=
U_f^T\delta^{a_f}_{t+1}
+
U_i^T\delta^{a_i}_{t+1}
+
U_o^T\delta^{a_o}_{t+1}
+
U_c^T\delta^{a_c}_{t+1}
$$

也就是說，$h_t$ 的梯度來自：
- 當前時間步的輸出層
- 下一個時間步各 gate 經 recurrent connection 回傳的梯度

### 11.2 Cell state 的總梯度（最關鍵）
$$
\delta^c_t =
\delta^h_t \odot o_t \odot \bigl(1-\tanh^2(c_t)\bigr)
+
\delta^c_{t+1}\odot f_{t+1}
$$

第二項讓梯度能沿著 cell state 長距離傳遞，這正是 LSTM 比普通 RNN 更不容易發生 gradient vanishing 的關鍵。

# GRU（閘門循環單元，Gated Recurrent Unit）

GRU 是一種循環神經網路（RNN）的變體，設計目的是在處理序列資料（如時間序列、語言）時，改善傳統 RNN 的「梯度消失（vanishing gradient）」問題，同時維持較低的模型複雜度。

## 核心概念

GRU 的關鍵在於「門控機制（gating mechanism）」，用來控制資訊在時間序列中的流動與保留。相比 LSTM，GRU 結構更簡化，僅包含兩個 gate：

1. **Update Gate（更新門，U）**
2. **Reset Gate（重設門，R）**

沒有獨立的 cell state（像 LSTM），而是直接用 hidden state 表示記憶。

GRU 相較於 LSTM 結構較簡單、參數較少，因此通常更省計算資源；可以視為對 LSTM 的簡化設計，同時仍保留 gate 機制來緩解傳統 RNN 的梯度消失問題，但在極長期依賴任務上可能略遜於 LSTM。

## 數學結構

在時間步 $t$，給定輸入 $x_t$ 與前一隱藏狀態 $h_{t-1}$：

1. Update Gate

控制「保留多少舊資訊」

$$
U_t = \sigma(x_t W_u + h_{t-1} W_{hu} + b_u)
$$

2. Reset Gate

控制「忽略多少舊資訊」

$$
R_t = \sigma(x_t W_r + h_{t-1} W_{hr} + b_r)
$$

3. Candidate Hidden State

利用 reset gate 決定如何使用舊狀態

$$
\tilde{h}_t = \tanh(x_t W_h + (R_t \odot h_{t-1}) W_{hh} + b_h)
$$

4. 最終 Hidden State 更新

在舊資訊與新資訊之間做加權

$$
h_t = U_t \odot h_{t-1} + (1 - U_t) \odot \tilde{h}_t
$$

## 直觀理解

可以用「記憶控制」來理解：

- **Update Gate（U）**
  - 接近 1 → 保留舊記憶（long-term memory）
  - 接近 0 → 使用新資訊

- **Reset Gate（R）**
  - 接近 0 → 忘記過去（適合新上下文）
  - 接近 1 → 保留過去資訊

## 與 LSTM 的差異

| 項目 | GRU | LSTM |
|------|-----|------|
| Gate 數量 | 2（U, R） | 3（input, forget, output） |
| Cell state | 無 | 有 |
| 計算量 | 較少 | 較多 |
| 表現 | 通常接近 LSTM | 稍強但較重 |

實務上：

- 資料量不大或需要快速訓練 → 常用 GRU
- 長期依賴很強 → LSTM 有時更穩定

## 優缺點總結

### 優點
- 結構簡單，參數較少  
- 訓練速度快  
- 能處理中長期依賴  

### 缺點
- 表達能力略低於 LSTM（部分任務）  
- gate 行為仍較難直觀解釋  

# GRU 的向後傳播推導

## 1. Forward 定義

在時間步 $t$，設：

- 輸入為 $x_t \in \mathbb{R}^{d_x}$
- 前一時刻隱藏狀態為 $h_{t-1} \in \mathbb{R}^{d_h}$
- 當前隱藏狀態為 $h_t \in \mathbb{R}^{d_h}$

GRU 的 forward 可寫成：

### (1) Update gate
$$
u_t = \sigma(a^u_t)
$$

其中

$$
a^u_t = x_t W_{xu} + h_{t-1} W_{hu} + b_u
$$

---

### (2) Reset gate
$$
r_t = \sigma(a^r_t)
$$

其中

$$
a^r_t = x_t W_{xr} + h_{t-1} W_{hr} + b_r
$$

---

### (3) Candidate hidden state
先定義

$$
m_t = r_t \odot h_{t-1}
$$

再令

$$
\tilde{h}_t = \tanh(a^h_t)
$$

其中

$$
a^h_t = x_t W_{xh} + m_t W_{hh} + b_h
$$

---

### (4) Hidden state update
$$
h_t = u_t \odot h_{t-1} + (1-u_t)\odot \tilde{h}_t
$$

---

若有輸出層，則：

$$
y_t = h_t W_y + b_y
$$

假設總 loss 為

$$
L = \sum_t L_t
$$

---


## 2. Backward 推導的重要梯度

在 backward 時，我們假設已知，並且這是推導的重要梯度之一：

$$
\frac{\partial L}{\partial h_t}
$$

這個梯度通常來自兩部分：

1. 目前時間步輸出層反傳回來的梯度
2. 下一個時間步透過 recurrent connection 傳回來的梯度

因此實作時常寫成：

$$
\delta^h_t = \frac{\partial L}{\partial h_t}
$$

---

## 3. 先對 hidden state 更新式反傳

由

$$
h_t = u_t \odot h_{t-1} + (1-u_t)\odot \tilde{h}_t
$$

可得：

### (1) 對 $\tilde{h}_t$ 的梯度
$$
\frac{\partial L}{\partial \tilde{h}_t}=
\frac{\partial L}{\partial h_t} \odot \frac{\partial h_t}{\partial \tilde{h}_t}=
\delta^h_t \odot (1-u_t)
$$

---

### (2) 對 $u_t$ 的梯度
$$
\frac{\partial L}{\partial u_t}=
\frac{\partial L}{\partial h_t} \odot \frac{\partial h_t}{\partial u_t}=
\delta^h_t \odot (h_{t-1} - \tilde{h}_t)
$$

---

### (3) 對 $h_{t-1}$ 的直接梯度
這裡是先只看 $h_t$ 公式中直接出現的 $h_{t-1}$：

$$
\left(\frac{\partial L}{\partial h_{t-1}}\right)_{\text{direct}}=
\delta^h_t \odot u_t
$$

注意：這還不是完整的 $\frac{\partial L}{\partial h_{t-1}}$，後面還要加上經過 $u_t$、$r_t$、$\tilde{h}_t$ 路徑回傳的部分。

---

## 4. Candidate hidden state 的反傳

由

$$
\tilde{h}_t = \tanh(a^h_t)
$$

可得：

$$
\frac{\partial L}{\partial a^h_t}=
\frac{\partial L}{\partial \tilde{h}_t}
\odot (1-\tilde{h}_t^2)
$$

令

$$
\delta^h_{a,t}=
\frac{\partial L}{\partial a^h_t}=
\left(\delta^h_t \odot (1-u_t)\right)\odot (1-\tilde{h}_t^2)
$$

---

## 5. 對 candidate 對應參數的梯度

因為

$$
a^h_t = x_t W_{xh} + m_t W_{hh} + b_h
$$

所以：

### (1) $W_{xh}$
$$
\frac{\partial L}{\partial W_{xh}}=
x_t^T \delta^h_{a,t}
$$

### (2) $W_{hh}$
$$
\frac{\partial L}{\partial W_{hh}}=
m_t^T \delta^h_{a,t}
$$

### (3) $b_h$
$$
\frac{\partial L}{\partial b_h}=
\sum_{\text{batch}} \delta^h_{a,t}
$$

---

## 6. 從 $a^h_t$ 往 $m_t$ 反傳

因為

$$
a^h_t = x_t W_{xh} + m_t W_{hh} + b_h
$$

所以

$$
\frac{\partial L}{\partial m_t}=
\delta^h_{a,t} W_{hh}^T
$$

而

$$
m_t = r_t \odot h_{t-1}
$$

因此可得：

### (1) 對 $r_t$ 的梯度
$$
\frac{\partial L}{\partial r_t}=
\frac{\partial L}{\partial m_t}\odot h_{t-1}
$$

也就是

$$
\frac{\partial L}{\partial r_t}=
(\delta^h_{a,t} W_{hh}^T)\odot h_{t-1}
$$

### (2) 對 $h_{t-1}$ 經由此路徑的梯度
$$
\left(\frac{\partial L}{\partial h_{t-1}}\right)_{\tilde{h}\text{-path}}=
(\delta^h_{a,t} W_{hh}^T)\odot r_t
$$

---

## 7. Reset gate 的反傳

由

$$
r_t = \sigma(a^r_t)
$$

可得

$$
\frac{\partial L}{\partial a^r_t}=
\frac{\partial L}{\partial r_t}\odot r_t \odot (1-r_t)
$$

令

$$
\delta^r_{a,t}=
\frac{\partial L}{\partial a^r_t}
$$

則

$$
\delta^r_{a,t}=
\left[(\delta^h_{a,t} W_{hh}^T)\odot h_{t-1}\right]\odot r_t(1-r_t)
$$

---

## 8. Reset gate 參數梯度

由

$$
a^r_t = x_t W_{xr} + h_{t-1} W_{hr} + b_r
$$

所以：

### (1) $W_{xr}$
$$
\frac{\partial L}{\partial W_{xr}}=
x_t^T \delta^r_{a,t}
$$

### (2) $W_{hr}$
$$
\frac{\partial L}{\partial W_{hr}}=
h_{t-1}^T \delta^r_{a,t}
$$

### (3) $b_r$
$$
\frac{\partial L}{\partial b_r}=
\sum_{\text{batch}} \delta^r_{a,t}
$$

並且由此再回傳到 $h_{t-1}$：

$$
\left(\frac{\partial L}{\partial h_{t-1}}\right)_{r\text{-path}}=
\delta^r_{a,t} W_{hr}^T
$$

---

## 9. Update gate 的反傳

由

$$
u_t = \sigma(a^u_t)
$$

可得

$$
\frac{\partial L}{\partial a^u_t}=
\frac{\partial L}{\partial u_t}\odot u_t(1-u_t)
$$

令

$$
\delta^u_{a,t}=
\frac{\partial L}{\partial a^u_t}
$$

則

$$
\delta^u_{a,t}=
\left[\delta^h_t \odot (h_{t-1}-\tilde{h}_t)\right]\odot u_t(1-u_t)
$$

---

## 10. Update gate 參數梯度

由

$$
a^u_t = x_t W_{xu} + h_{t-1} W_{hu} + b_u
$$

所以：

### (1) $W_{xu}$
$$
\frac{\partial L}{\partial W_{xu}}=
x_t^T \delta^u_{a,t}
$$

### (2) $W_{hu}$
$$
\frac{\partial L}{\partial W_{hu}}=
h_{t-1}^T \delta^u_{a,t}
$$

### (3) $b_u$
$$
\frac{\partial L}{\partial b_u}=
\sum_{\text{batch}} \delta^u_{a,t}
$$

同時由此對 $h_{t-1}$ 再有一條路徑：

$$
\left(\frac{\partial L}{\partial h_{t-1}}\right)_{u\text{-path}}=
\delta^u_{a,t} W_{hu}^T
$$

---

## 11. 合併對 $h_{t-1}$ 的總梯度

因此前一時刻隱藏狀態的總梯度為：

$$
\frac{\partial L}{\partial h_{t-1}}=
\delta^h_t \odot u_t
+
(\delta^h_{a,t} W_{hh}^T)\odot r_t
+
\delta^r_{a,t} W_{hr}^T
+
\delta^u_{a,t} W_{hu}^T
$$

這就是時間反向傳播時要傳給前一個時間步的 $dH_{\text{next}}$。

---

## 12. 對輸入 $x_t$ 的梯度

如果還需要往更前面的網路層反傳，則：

$$
\frac{\partial L}{\partial x_t}=
\delta^u_{a,t} W_{xu}^T
+
\delta^r_{a,t} W_{xr}^T
+
\delta^h_{a,t} W_{xh}^T
$$

---

## 13. 若有輸出層的梯度

若

$$
y_t = h_t W_y + b_y
$$

且已知輸出層 loss 對 logits 的梯度為

$$
\delta^z_t = \frac{\partial L}{\partial y_t}
$$

則：

### (1) 輸出層參數梯度
$$
\frac{\partial L}{\partial W_y}=
h_t^T \delta^z_t
$$

$$
\frac{\partial L}{\partial b_y}=
\sum_{\text{batch}} \delta^z_t
$$

### (2) 從輸出層回到 hidden state
$$
\left(\frac{\partial L}{\partial h_t}\right)_{\text{output}}=
\delta^z_t W_y^T
$$

若再加上下一時間步傳回來的梯度 $\delta^{\text{next}}_t$，則

$$
\delta^h_t=
\delta^z_t W_y^T + \delta^{\text{next}}_t
$$

---



## 14. 對照程式實作的對應關係

若對照你前面的 NumPy 實作：

- `dH = np.dot(dZ, Wy.T) + dH_next`
  - 對應：
  $$
  \delta^h_t = \delta^z_t W_y^T + \delta^{\text{next}}_t
  $$

- `dH_tilda = dH * (1 - U)`
  - 對應：
  $$
  \frac{\partial L}{\partial \tilde{h}_t}
  =
  \delta^h_t \odot (1-u_t)
  $$

- `dU = H_pre * dH - H_tilda * dH`
  - 對應：
  $$
  \frac{\partial L}{\partial u_t}
  =
  \delta^h_t \odot (h_{t-1} - \tilde{h}_t)
  $$

- `dH_tildaZ = (1 - np.square(H_tilda)) * dH_tilda`
  - 對應：
  $$
  \delta^h_{a,t}
  =
  \left(\delta^h_t \odot (1-u_t)\right)\odot (1-\tilde{h}_t^2)
  $$

- `dR = np.dot(dH_tildaZ, Whh.T) * H_pre`
  - 對應：
  $$
  \frac{\partial L}{\partial r_t}
  =
  (\delta^h_{a,t} W_{hh}^T)\odot h_{t-1}
  $$

- `dUZ = U * (1-U) * dU`
  - 對應：
  $$
  \delta^u_{a,t}
  =
  \frac{\partial L}{\partial u_t}\odot u_t(1-u_t)
  $$

- `dRZ = R * (1-R) * dR`
  - 對應：
  $$
  \delta^r_{a,t}
  =
  \frac{\partial L}{\partial r_t}\odot r_t(1-r_t)
  $$

---

## 15. 最後整理：單一時間步的 backward 公式

給定 $\delta^h_t = \frac{\partial L}{\partial h_t}$：

### Step 1
$$
\delta^{\tilde{h}}_t = \delta^h_t \odot (1-u_t)
$$

$$
\delta^u_t = \delta^h_t \odot (h_{t-1} - \tilde{h}_t)
$$

$$
\delta^{h\to h_{t-1}}_{\text{direct}} = \delta^h_t \odot u_t
$$

### Step 2
$$
\delta^h_{a,t} = \delta^{\tilde{h}}_t \odot (1-\tilde{h}_t^2)
$$

### Step 3
$$
\delta^m_t = \delta^h_{a,t} W_{hh}^T
$$

$$
\delta^r_t = \delta^m_t \odot h_{t-1}
$$

$$
\delta^{h\to h_{t-1}}_{\text{cand}} = \delta^m_t \odot r_t
$$

### Step 4
$$
\delta^r_{a,t} = \delta^r_t \odot r_t(1-r_t)
$$

$$
\delta^u_{a,t} = \delta^u_t \odot u_t(1-u_t)
$$

### Step 5
$$
\frac{\partial L}{\partial h_{t-1}}=
\delta^{h\to h_{t-1}}_{\text{direct}}
+
\delta^{h\to h_{t-1}}_{\text{cand}}
+
\delta^r_{a,t} W_{hr}^T
+
\delta^u_{a,t} W_{hu}^T
$$

---

## 16. 直觀理解

GRU backward 的本質是把 $h_t$ 對過去的依賴拆成四條路徑：

1. **直接路徑**：$h_t \leftarrow h_{t-1}$
2. **candidate 路徑**：$h_t \leftarrow \tilde{h}_t \leftarrow h_{t-1}$
3. **reset gate 路徑**：$h_t \leftarrow \tilde{h}_t \leftarrow r_t \leftarrow h_{t-1}$
4. **update gate 路徑**：$h_t \leftarrow u_t \leftarrow h_{t-1}$

所以在實作時，$dH_{pre}$ 會是這幾條路徑加總後的結果，而不是單一路徑。

---